# 03 — Grouping & Aggregation

Grouped operations, pivoting, cube/rollup, and the `df.stat` namespace. All aggregation runs inside IRIS via SQL pushdown.

In [ ]:
import os
from dotenv import load_dotenv
from irispark import IrisParkSession

load_dotenv()

# Connection via environment variables (matches examples/basic_usage.py).
# Set IRIS_HOST / IRIS_PORT / IRIS_NAMESPACE / IRIS_USERNAME / IRIS_PASSWORD.
try:
    session = IrisParkSession.builder() \
        .host(os.environ.get("IRIS_HOST", "localhost")) \
        .port(int(os.environ.get("IRIS_PORT", 1972))) \
        .namespace(os.environ.get("IRIS_NAMESPACE", "USER")) \
        .username(os.environ.get("IRIS_USERNAME", "_SYSTEM")) \
        .password(os.environ.get("IRIS_PASSWORD", "SYS")) \
        .getOrCreate()
    print("Connected to IRIS:", session)
except Exception as e:
    print("SKIP: IRIS not reachable -", e)
    session = None

In [ ]:
if session is None:
    raise SystemExit("IRIS not reachable; skipping this notebook.")

## 1. `groupBy` + `agg`

Aggregate by state and city.

In [ ]:
from irispark.functions import sum as s, avg, count, min, max, stddev, corr

df = session.table("vendas")

df.groupBy("estado").agg(
    s("valor").alias("total"),
    avg("valor").alias("media"),
    count("id").alias("n"),
    min("valor").alias("min"),
    max("valor").alias("max"),
    stddev("valor").alias("std"),
).orderBy("total DESC").show()

## 2. `GroupedData` convenience methods

In [ ]:
g = df.groupBy("estado")
g.sum("valor").show()
g.avg("valor").show()
g.mean("valor").show()
g.count().show()
g.min("valor").show()
g.max("valor").show()

## 3. Pivot

Pivot the city dimension by state.

In [ ]:
df.groupBy("cidade").pivot("estado").agg(s("valor")).show()

## 4. Cube & Rollup

In [ ]:
df.cube("estado", "cidade").agg(s("valor")).show()
df.rollup("estado", "cidade").agg(s("valor")).show()

## 5. `df.stat` namespace

Correlation, covariance, crosstab, frequent items, and stratified sampling.

In [ ]:
print("corr(id, valor):", df.stat.corr("id", "valor"))
print("cov(id, valor):", df.stat.cov("id", "valor"))
print("crosstab(estado, cidade):")
df.stat.crosstab("estado", "cidade")

## 6. `freqItems` and `sampleBy`

`freqItems` returns a dict of value counts; `sampleBy` returns a pandas DataFrame (stratified sample).

In [ ]:
print("freqItems:", df.stat.freqItems())

sampled = df.stat.sampleBy("estado", {"SP": 0.5, "RJ": 0.5, "MG": 0.5}, sampleByColumns=["estado"])
print("sampleBy type:", type(sampled).__name__)
print("sampleBy rows:", len(sampled))
sampled.head()

## 7. SQL transparency

In [ ]:
print(df.groupBy("estado").agg(s("valor").alias("total")).to_sql())

In [ ]:
if session is not None:
    session.close()
    print("Session closed.")